# RAG Pipeline for MCQ Solver  — Kaggle Edition

## What is RAG?
**Retrieval-Augmented Generation (RAG)** enhances answer prediction by first *retrieving* relevant
external documents (our Wikipedia corpus) and then using that retrieved context to *rank* the MCQ options.

## Kaggle Setup Instructions

Before running this notebook you need to attach **two datasets** from the Kaggle sidebar → *Data* → *+ Add Data*:

| What | Kaggle Dataset slug | Path inside notebook |
|------|--------------------|----------------------|
| Competition data (train/test CSVs) | the competition dataset | `/kaggle/input/<competition-slug>/` |
| Wikipedia corpus (.md files) | your uploaded wiki corpus dataset | `/kaggle/input/<your-wiki-dataset-slug>/wiki_corpus/` |

> **How to upload the wiki corpus to Kaggle:**  
> Kaggle Datasets → New Dataset → upload the `wiki_corpus/` folder as a zip → use the slug here.

Edit the two `*_SLUG` constants in the **Setup** cell below to match your actual Kaggle dataset slugs.

## Pipeline Overview
```
OFFLINE (done once per session)
  1. Load all .md files from the wiki corpus dataset
  2. Chunk each article into overlapping 500-char passages
  3. Fit a TF-IDF vectorizer  →  sparse (chunks x vocab) matrix

ONLINE (per question)
  4. Embed the question prompt  →  TF-IDF query vector
  5. Retrieve top-K most similar chunks (cosine similarity)
  6. Score each option: cosine_sim(context, prompt + option)
  7. Rank options → top-3 predictions  →  submission CSV
```

## Notebook Sections
1. Environment Detection and Path Setup
2. Install / Import Dependencies
3. Load and Explore the MCQ Dataset
4. Load and Chunk the Wikipedia Corpus
5. Build the TF-IDF Retrieval Index
6. Define the Retrieval Function
7. Define the Scoring and Ranking Function
8. Evaluate on Training Set (MAP@3)
9. Generate and Save the Submission File
10. Bonus — Dense Retrieval with Sentence-Transformers (GPU)

## 1. Environment Detection and Path Setup

Kaggle notebooks run inside a container where:
- All **input datasets** live under `/kaggle/input/<dataset-slug>/`
- All **output files** (including `submission.csv`) must go to `/kaggle/working/`
- The notebook does **not** have a `__file__` attribute, so we detect Kaggle
  via `os.path.exists('/kaggle')` and set paths accordingly.

**Edit the two slug constants below to match your Kaggle dataset names.**

In [ ]:
import os

# ── Kaggle Dataset Slugs ───────────────────────────────────────────────────────
# Replace these with the actual slug names of your Kaggle datasets.
# You can find the slug in the dataset URL:  kaggle.com/datasets/<user>/<slug>

COMPETITION_SLUG = "dl-gen-ai-may-2026"       # competition dataset slug
WIKI_CORPUS_SLUG = "mcq-wiki-corpus"           # your uploaded wiki corpus slug

# ── Environment Detection ─────────────────────────────────────────────────────
# Detect whether we are running on Kaggle or locally.
# This allows the same notebook to work in both environments.
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    # Kaggle paths
    COMPETITION_DIR = f"/kaggle/input/{COMPETITION_SLUG}"
    CORPUS_DIR      = f"/kaggle/input/{WIKI_CORPUS_SLUG}/wiki_corpus"
    OUTPUT_DIR      = "/kaggle/working"              # Kaggle output folder
else:
    # Local paths (for development on your own machine)
    _this_dir       = os.path.dirname(os.path.abspath("__file__"))
    COMPETITION_DIR = _this_dir
    CORPUS_DIR      = os.path.join(_this_dir, "wiki_corpus")
    OUTPUT_DIR      = _this_dir

TRAIN_PATH  = os.path.join(COMPETITION_DIR, "train.csv")
TEST_PATH   = os.path.join(COMPETITION_DIR, "test.csv")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

print(f"Running on Kaggle : {ON_KAGGLE}")
print(f"Corpus directory  : {CORPUS_DIR}")
print(f"Train CSV         : {TRAIN_PATH}")
print(f"Test  CSV         : {TEST_PATH}")
print(f"Submission output : {OUTPUT_PATH}")

# Quick sanity check — make sure the paths actually exist before continuing
assert os.path.exists(CORPUS_DIR),   f"Wiki corpus not found at: {CORPUS_DIR}\nCheck your WIKI_CORPUS_SLUG"
assert os.path.exists(TRAIN_PATH),   f"train.csv not found at:   {TRAIN_PATH}\nCheck your COMPETITION_SLUG"
assert os.path.exists(TEST_PATH),    f"test.csv not found at:    {TEST_PATH}\nCheck your COMPETITION_SLUG"
print("\nAll paths verified OK!")

Running on Kaggle : False
Corpus directory  : /Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/DL_GenAI_May_2026/RAG/wiki_corpus
Train CSV         : /Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/DL_GenAI_May_2026/RAG/train.csv
Test  CSV         : /Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/DL_GenAI_May_2026/RAG/test.csv
Submission output : /Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/DL_GenAI_May_2026/RAG/submission.csv

All paths verified OK!


## 2. Install and Import Dependencies

Kaggle already pre-installs `numpy`, `pandas`, `scikit-learn`, and `tqdm`.
We use `tqdm.auto` which automatically picks the best progress bar style
(notebook-style on Kaggle/Jupyter, plain text in terminal).

In [100]:
# All of these are pre-installed on Kaggle; no !pip needed.
# If running locally and a package is missing: pip install scikit-learn pandas tqdm numpy

import re
import glob
import numpy as np
import pandas as pd
from collections import defaultdict

# tqdm.auto picks notebook-style progress bars when inside Jupyter/Kaggle
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print(f"numpy   version: {np.__version__}")
print(f"pandas  version: {pd.__version__}")
print("All imports successful.")

numpy   version: 2.2.0
pandas  version: 2.3.0
All imports successful.


## 3. Load and Explore the MCQ Dataset

Each row is one MCQ question:
- `id` — unique question ID
- `prompt` — question text (sometimes prefixed with a template like "Pick the best possible answer:")
- `A`, `B`, `C`, `D`, `E` — five candidate answer options
- `answer` — ground-truth correct label (only in `train.csv`)

The competition is evaluated using **MAP@3** — we need to predict the top-3 most likely correct answers.

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train rows: {len(train_df):,}  |  Columns: {list(train_df.columns)}")
print(f"Test  rows: {len(test_df):,}   |  Columns: {list(test_df.columns)}")
print()

train_df.head(3)

Train rows: 2,000  |  Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test  rows: 500   |  Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']



,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [102]:
# Answer distribution in training data
# Knowing which labels appear most often lets us build a smart fallback:
# if the RAG model is uncertain, default to globally most common answers.
print("Answer label distribution:")
print(train_df["answer"].value_counts())

# These 3 labels are used as a fallback if retrieval scores are all zero
GLOBAL_TOP3 = train_df["answer"].value_counts().index[:3].tolist()
print(f"\nGlobal top-3 fallback: {GLOBAL_TOP3}")

Answer label distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Global top-3 fallback: ['B', 'C', 'A']


## 4. Load and Chunk the Wikipedia Corpus

### Why Chunking?
A Wikipedia article can be 50,000+ characters long. Embedding the whole article as one unit
would drown out the specific paragraph that actually answers a given question.

**Sliding-window chunking** splits each article into smaller overlapping passages:
- **chunk_size = 500 chars** ≈ one paragraph
- **overlap = 100 chars** — the last 100 chars of each chunk are repeated at the
  start of the next, so sentences are never cut off at a chunk boundary

Each chunk is stored as `{"text": ..., "source": filename}` for later explainability.

In [103]:
def load_and_chunk_corpus(
    corpus_dir: str,
    chunk_size: int = 500,
    overlap: int = 100
) -> list:
    """
    Walk through all .md files in corpus_dir, clean the markdown,
    and split each article into overlapping fixed-size character chunks.

    Parameters
    ----------
    corpus_dir : str  — folder containing .md Wikipedia articles
    chunk_size : int  — max characters per chunk
    overlap    : int  — characters of overlap between consecutive chunks

    Returns
    -------
    List of dicts, each with keys 'text' (str) and 'source' (str).
    """
    chunks = []
    md_files = glob.glob(os.path.join(corpus_dir, "*.md"))
    print(f"Found {len(md_files)} .md files in corpus.")

    for filepath in tqdm(md_files, desc="Chunking corpus"):
        filename = os.path.basename(filepath)

        with open(filepath, "r", encoding="utf-8") as f:
            raw = f.read()

        # ── Clean the Markdown ───────────────────────────────────────────────
        # Strip heading markers (## Introduction) but keep the heading text
        text = re.sub(r"^#{1,6}\s+", "", raw, flags=re.MULTILINE)
        # Drop HTML/XML tags (e.g. <ref>, <math>)
        text = re.sub(r"<[^>]+>", "", text)
        # Drop front-matter blockquote lines added by our fetcher
        # (lines starting with "> **Query Topic**:", "> **Wikipedia Page**:", etc.)
        text = re.sub(r"^>.*$", "", text, flags=re.MULTILINE)
        # Collapse runs of blank lines
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        if not text:
            continue  # skip empty or near-empty files

        # ── Sliding-Window Chunking ─────────────────────────────────────────
        # Advance by (chunk_size - overlap) each step so adjacent chunks share
        # 'overlap' characters, preventing sentences from being cut at boundaries.
        step = chunk_size - overlap
        for start in range(0, len(text), step):
            chunk = text[start : start + chunk_size]
            if len(chunk.strip()) > 50:   # skip trivially short tail chunks
                chunks.append({"text": chunk, "source": filename})

    return chunks


all_chunks = load_and_chunk_corpus(CORPUS_DIR, chunk_size=500, overlap=200)
chunk_texts = [c["text"] for c in all_chunks]   # plain text list for TF-IDF

print(f"\nTotal chunks: {len(all_chunks):,}")
print(f"\nSample chunk from '{all_chunks[100]['source']}':")
print(all_chunks[100]["text"][:300])

Found 587 .md files in corpus.


Chunking corpus: 100%|██████████| 587/587 [00:00<00:00, 1600.17it/s]



Total chunks: 62,971

Sample chunk from 'peierls.md':
aper.
Heisenberg left in 1929 to lecture in America, China, Japan and India, and on his recommendation Peierls moved on to ETH Zurich, where he studied under Wolfgang Pauli. Pauli set him a problem of investigating the vibration of atoms in a crystal lattice. Peierls explored—and named—the phenomeno


## 5. Build the TF-IDF Retrieval Index

### What is TF-IDF?
**Term Frequency – Inverse Document Frequency** converts text into sparse numerical vectors:
- **TF (Term Frequency)**: how often a word appears in this chunk
- **IDF (Inverse Document Frequency)**: penalises words that appear in almost every chunk
  (e.g. "the", "is") — these carry little discriminative power

The result is a sparse matrix: **rows = chunks, columns = vocabulary terms**.

### Why TF-IDF over dense embeddings (for now)?
| | TF-IDF | Dense (SBERT) |
|---|---|---|
| Speed | Very fast (ms/query) | Moderate (needs GPU) |
| Memory | Sparse, ~efficient | Dense float32 array |
| Kaggle RAM | Fits in 16 GB | Needs GPU memory |
| Semantic matching | Exact words only | Paraphrases too |

Section 10 shows the dense upgrade using `sentence-transformers`.

In [104]:
# Build the TF-IDF index over all corpus chunks.
#
# Parameter explanations:
#   sublinear_tf=True  → use log(1+tf) instead of raw tf to reduce dominance
#                         of very frequent terms within a chunk
#   min_df=2           → drop terms appearing in fewer than 2 chunks
#                         (likely typos, math symbols, or one-off tokens)
#   max_df=0.95        → drop terms in 95%+ of chunks (so common they're noise)
#   ngram_range=(1,2)  → include unigrams ('Einstein') AND bigrams ('dark matter')
#                         to capture multi-word scientific concepts
#   stop_words         → remove high-frequency English function words
vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    stop_words="english",
)

print("Fitting TF-IDF vectorizer on all chunks (this takes ~20-30 s for 47K chunks)...")
corpus_matrix = vectorizer.fit_transform(chunk_texts)  # shape: (num_chunks, vocab_size)

vocab_size  = corpus_matrix.shape[1]
num_chunks  = corpus_matrix.shape[0]
sparsity    = 1 - corpus_matrix.nnz / (num_chunks * vocab_size)

print(f"\nCorpus matrix : {num_chunks:,} chunks  x  {vocab_size:,} vocab terms")
print(f"Sparsity      : {sparsity:.4f}  (most entries are zero — that's expected)")

Fitting TF-IDF vectorizer on all chunks (this takes ~20-30 s for 47K chunks)...

Corpus matrix : 62,971 chunks  x  535,800 vocab terms
Sparsity      : 0.9999  (most entries are zero — that's expected)


## 6. Define the Retrieval Function

Given a question prompt, we:
1. Transform it with the **already-fitted** vectorizer (`transform`, not `fit_transform`)
2. Compute cosine similarity between the query vector and all chunk vectors
3. Return the top-K highest-scoring chunks

**Cosine similarity** measures the angle between two vectors — 1.0 = identical direction (most relevant), 0.0 = no shared terms.

In [105]:
def retrieve_top_k_chunks(query: str, top_k: int = 5) -> list:
    """
    Retrieve the top_k most relevant Wikipedia chunks for a given query.

    Parameters
    ----------
    query : str  — the question text to search for
    top_k : int  — number of chunks to return

    Returns
    -------
    List of chunk text strings ordered by descending relevance.
    """
    # transform() reuses the vocabulary fixed during fit_transform() above.
    # NEVER call fit_transform() again here — that would create a new vocabulary.
    query_vec = vectorizer.transform([query])           # shape: (1, vocab_size)

    # cosine_similarity returns a (1, num_chunks) matrix; .flatten() → 1-D array
    scores = cosine_similarity(query_vec, corpus_matrix).flatten()

    # np.argpartition is O(n) and much faster than a full sort when we only
    # need the top-K indices (full sort is O(n log n))
    if top_k >= len(scores):
        top_indices = np.argsort(scores)[::-1]
    else:
        top_indices = np.argpartition(scores, -top_k)[-top_k:]
        top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

    return [all_chunks[i]["text"] for i in top_indices]


# Quick sanity check
demo_query = "What is the photoelectric effect?"
retrieved  = dense_retrieve_top_k(demo_query, top_k=3)

print(f"Query: '{demo_query}'\n")
for rank, chunk in enumerate(retrieved, 1):
    print(f"── Rank #{rank} ──")
    print(chunk[:250])
    print()

/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: divide by zero encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: overflow encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: invalid value encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)


Query: 'What is the photoelectric effect?'

── Rank #1 ──
he renovation of the existing buildings, this will also require the construction of a new central humanities library in the inner courtyard and another new building in the immediate vicinity of the Himbeerpalast. In addition, the old Chemikum buildin

── Rank #2 ──
 hall building for the philosophy faculty.
The existing buildings of the Faculty of Philosophy in the north of Erlangen are to be demolished, with the exception of the Juridicum, to make way for the nearby University Hospital and the Faculty of Medic

── Rank #3 ──
ion to the direct costs. Indirect costs include overhead, profit, sales or use taxes, payment and performance bonds, escalation, and contingency. Profit is cost to the buyer, but is not a cost to the provider, rather a projection of anticipated incom



## 7. Define the Scoring and Ranking Function

Once we have the retrieved context, we score each answer option (A–E) by measuring how similar
the combined `prompt + option` text is to the retrieved Wikipedia context.

The option with the **highest cosine similarity to the context** is ranked first.

```
context_vec  = TF-IDF( chunk_1 + chunk_2 + ... + chunk_K )

for each option in [A, B, C, D, E]:
    hypothesis_vec  = TF-IDF( prompt + " " + option_text )
    score[option]   = cosine_sim(context_vec, hypothesis_vec)

prediction = top 3 options sorted by score descending
```

In [106]:
OPTION_COLS = ["A", "B", "C", "D", "E"]

def rag_predict_row(row: dict, top_k_retrieval: int = 5) -> list:
    """
    Full RAG prediction for a single MCQ row.

    Steps:
      1. Retrieve top_k Wikipedia chunks relevant to the question
      2. Concatenate them into a context string
      3. Score each option: cosine_sim(context, prompt+option)
      4. Return top-3 option labels in descending score order

    Parameters
    ----------
    row             : dict — one MCQ row (keys: 'prompt', 'A', 'B', 'C', 'D', 'E')
    top_k_retrieval : int  — number of Wikipedia chunks to retrieve

    Returns
    -------
    List of 3 option label strings, e.g. ['B', 'C', 'A']
    """
    prompt = str(row["prompt"])

    # ── Step 1: Retrieve ──────────────────────────────────────────────────────
    retrieved_chunks = dense_retrieve_top_k(prompt, top_k=top_k_retrieval)

    if not retrieved_chunks:
        # Safety fallback: if nothing is retrieved, use the globally most
        # frequent answer labels (roughly correct ~42% of the time)
        return GLOBAL_TOP3[:]

    # ── Step 2: Build Context ────────────────────────────────────────────────
    # Join retrieved chunks into one text blob that acts as our "knowledge"
    # for this question.  Space-joining is sufficient for TF-IDF.
    context = " ".join(retrieved_chunks)
    context_vec = vectorizer.transform([context])   # shape: (1, vocab_size)

    # ── Step 3: Score Each Option ────────────────────────────────────────────
    scores = []
    for opt in OPTION_COLS:
        # The hypothesis is the prompt combined with this candidate answer.
        # A correct answer should match the retrieved Wikipedia context well.
        hypothesis     = prompt + " " + str(row[opt])
        hypothesis_vec = vectorizer.transform([hypothesis])
        sim = cosine_similarity(context_vec, hypothesis_vec)[0][0]
        scores.append(sim)

    # ── Step 4: Rank and Return Top-3 ───────────────────────────────────────
    scores_arr     = np.array(scores)
    sorted_indices = np.argsort(-scores_arr, kind="stable")  # descending
    return [OPTION_COLS[i] for i in sorted_indices[:3]]


# Quick demo on the first training row
sample = train_df.iloc[0].to_dict()
pred   = rag_predict_row(sample, top_k_retrieval=5)

print("── RAG Demo ──")
print(f"Prompt     : {sample['prompt'][:90]}...")
print(f"Prediction : {pred}")
print(f"Correct    : {sample['answer']}")

/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: divide by zero encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: overflow encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: invalid value encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)


── RAG Demo ──
Prompt     : Pick the best possible answer: What is Martin Heidegger's view on the relationship between...
Prediction : ['A', 'B', 'C']
Correct    : B


## 8. Evaluate on Training Set (MAP@3)

### What is MAP@3?
The competition scores submissions with **Mean Average Precision at 3 (MAP@3)**.

For a single question with correct answer `X` and predictions `[P1, P2, P3]`:

| Prediction list | Score |
|----------------|-------|
| `[X, ?, ?]` | 1.0 (correct first) |
| `[?, X, ?]` | 0.5 (correct second) |
| `[?, ?, X]` | 0.33 (correct third) |
| `[?, ?, ?]` | 0.0 (not in top-3) |

**MAP@3** = average of these scores over all questions in the test set.

In [ ]:
def apk(actual: str, predicted: list, k: int = 3) -> float:
    """
    Average Precision at K for a single question.

    Parameters
    ----------
    actual    : str   — ground truth label, e.g. 'B'
    predicted : list  — model predictions, e.g. ['A', 'B', 'C']
    k         : int   — cutoff (3 for this competition)

    Returns
    -------
    float in [0, 1]
    """
    predicted = predicted[:k]
    score, num_hits = 0.0, 0.0
    for i, p in enumerate(predicted):
        # Count as a hit if the prediction matches and hasn't appeared earlier
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)   # precision at position i+1
    return score / min(1, k)


def mapk(actuals: list, all_preds: list, k: int = 3) -> float:
    """Mean Average Precision at K over all questions."""
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, all_preds)]))

In [11]:
# ── Evaluate on a sample of training data ─────────────────────────────────────
# Evaluating all 2000 rows is slow on CPU (~20 minutes).
# Start with N_EVAL=200 to get a quick directional score.
# Change to len(train_df) for the full training-set score before submitting.
N_EVAL = 2000

eval_df     = train_df.head(N_EVAL).reset_index(drop=True)
predictions = []
actuals     = []

for _, row in tqdm(eval_df.iterrows(), total=N_EVAL, desc=f"Eval ({N_EVAL} rows)"):
    predictions.append(rag_predict_row(row.to_dict(), top_k_retrieval=5))
    actuals.append(row["answer"])

score = mapk(actuals, predictions, k=3)
print(f"\nMAP@3 on first {N_EVAL} training rows: {score:.5f}")

Eval (2000 rows):   0%|          | 0/2000 [00:00<?, ?it/s]/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: divide by zero encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: overflow encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: invalid value encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
Eval (2000 rows): 100%|██████████| 2000/2000 [00:15<00:00, 125.13it/s]



MAP@3 on first 2000 training rows: 0.30117


In [109]:
# Per-label breakdown — reveals if the model is biased toward certain options
per_label = defaultdict(list)
for actual, pred in zip(actuals, predictions):
    per_label[actual].append(apk(actual, pred, k=3))

print(f"{'Label':<8} {'Count':<8} {'Mean AP@3'}")
print("-" * 30)
for label in sorted(per_label):
    vals = per_label[label]
    print(f"{label:<8} {len(vals):<8} {np.mean(vals):.4f}")

Label    Count    Mean AP@3
------------------------------
A        369      0.4598
B        490      0.3459
C        459      0.3257
D        358      0.1904
E        324      0.1404


## 9. Generate and Save the Submission File

We run RAG inference on the full test set and write `submission.csv` to `/kaggle/working/`.

Kaggle automatically makes files in `/kaggle/working/` available for download and submission.

Required format:
```
ID,Prediction
1,A B C
2,C A D
```

In [110]:
print(f"Running RAG on {len(test_df):,} test rows...")

rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test inference"):
    pred = rag_predict_row(row.to_dict(), top_k_retrieval=5)
    rows.append({
        "ID"         : row["id"],
        "Prediction" : " ".join(pred)   # e.g. 'B C A'
    })

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSubmission saved → {OUTPUT_PATH}")
print(f"Shape: {submission_df.shape}")
submission_df.head(10)

Running RAG on 500 test rows...


Test inference:   0%|          | 0/500 [00:00<?, ?it/s]/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: divide by zero encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: overflow encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: invalid value encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
Test inference: 100%|██████████| 500/500 [00:03<00:00, 144.75it/s]



Submission saved → /Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/DL_GenAI_May_2026/RAG/submission.csv
Shape: (500, 2)


,ID,Prediction
0,1,D A E
1,2,D A C
2,3,C A E
3,4,D B A
4,5,A B C
5,6,A B C
6,7,B A E
7,8,A B C
8,9,A B C
9,10,C A D


## 10. Bonus — Dense Retrieval with Sentence-Transformers (GPU)

TF-IDF only matches on **exact shared words**. A question about
*"Einstein's 1905 paper on light quanta"* may miss chunks about the *"photoelectric effect"*
because they share few surface words.

**Dense retrieval** uses a pre-trained neural model to embed text into a
384-dimensional semantic space — similar *meanings* get similar vectors.

### Kaggle GPU Setup
- Go to **Session options → Accelerator → GPU T4 x2** (free on Kaggle)
- `sentence-transformers` is pre-installed on Kaggle GPU images
- Embedding ~47K chunks takes **≈90 s on GPU** vs. ≈10 min on CPU

### Save Embeddings to Avoid Recomputing
Embeddings are saved to `/kaggle/working/chunk_embeddings.npy` so you can
load them in a follow-up session without re-running the model.

In [111]:
# ── Dense Retrieval Toggle ─────────────────────────────────────────────────────
# Set to True when running on a Kaggle GPU session.
# sentence-transformers is pre-installed on Kaggle; no !pip needed.
DENSE_ENABLED = True

if DENSE_ENABLED:
    import torch
    from sentence_transformers import SentenceTransformer

    # Detect GPU availability (Kaggle GPU sessions have CUDA)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # ── Load Model ────────────────────────────────────────────────────────────
    # all-MiniLM-L6-v2: 22 MB, 384-dim embeddings, fast and accurate.
    # Better alternatives (larger but slower):
    #   'all-mpnet-base-v2'          (768-dim, highest quality)
    #   'multi-qa-MiniLM-L6-cos-v1'  (optimised for Q&A retrieval)
    MODEL_NAME = "all-MiniLM-L6-v2"
    print(f"Loading model: {MODEL_NAME}")
    sbert = SentenceTransformer(MODEL_NAME, device=device)

    # ── Embed Corpus (with disk caching) ─────────────────────────────────────
    # Embeddings are saved to /kaggle/working/ so they survive across sessions.
    EMB_PATH = os.path.join(OUTPUT_DIR, "chunk_embeddings.npy")

    if os.path.exists(EMB_PATH):
        print(f"Loading pre-computed embeddings from {EMB_PATH}")
        chunk_embs = np.load(EMB_PATH)
    else:
        print(f"Embedding {len(chunk_texts):,} chunks on {device}...")
        chunk_embs = sbert.encode(
            chunk_texts,
            batch_size=256,              # increase if you have a large GPU
            show_progress_bar=True,
            normalize_embeddings=True,   # L2-normalise → cosine sim = dot product
            device=device,
        )
        np.save(EMB_PATH, chunk_embs)
        print(f"Embeddings saved → {EMB_PATH}")

    print(f"Embedding matrix shape: {chunk_embs.shape}")

    def dense_retrieve_top_k(query: str, top_k: int = 5) -> list:
        """
        Dense retrieval: encode the query with the same SBERT model and
        find the top_k nearest chunk embeddings via dot product
        (equivalent to cosine similarity because both sides are L2-normalised).
        """
        q_emb  = sbert.encode([query], normalize_embeddings=True, device=device)
        scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)

        if top_k >= len(scores):
            top_idx = np.argsort(scores)[::-1]
        else:
            top_idx = np.argpartition(scores, -top_k)[-top_k:]
            top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

        return [all_chunks[i]["text"] for i in top_idx]

    # ── Quick demo ────────────────────────────────────────────────────────────
    print("\n── Dense Retrieval Demo ──")
    for rank, chunk in enumerate(dense_retrieve_top_k("Heisenberg uncertainty principle", top_k=2), 1):
        print(f"Rank #{rank}: {chunk[:220]}...\n")

    print("To use dense retrieval everywhere, replace retrieve_top_k_chunks")
    print("with dense_retrieve_top_k inside rag_predict_row().")

else:
    print("Dense retrieval is OFF.")
    print("Set DENSE_ENABLED = True and enable a Kaggle GPU session to run this section.")

Using device: cpu
Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19795.33it/s]


Loading pre-computed embeddings from /Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/DL_GenAI_May_2026/RAG/chunk_embeddings.npy
Embedding matrix shape: (47298, 384)

── Dense Retrieval Demo ──


/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: divide by zero encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: overflow encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_45895/2620377792.py:51: RuntimeWarning: invalid value encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)


Rank #1: t Theology, Jurisprudence, Medicine and Philosophy.
1769: The Universität Erlangen is given the new name of Friedrich-Alexander-Universität in honour of Alexander, Margrave of Ansbach and Bayreuth.
1818: The library of t...

Rank #2:  that continuous positive net exchange of energy from the Casimir effect was possible, even stating in the abstract "In the event of no other alternative explanations, one should conclude that major technological advance...

To use dense retrieval everywhere, replace retrieve_top_k_chunks
with dense_retrieve_top_k inside rag_predict_row().


## Summary

### What We Built

| Step | Component | Notes |
|------|-----------|-------|
| Path setup | Kaggle-aware (`/kaggle/input/`, `/kaggle/working/`) | Edit slugs in Section 1 |
| Corpus loading | 587 Wikipedia `.md` files → 47K chunks | Sliding window, 500 chars |
| TF-IDF index | Sparse (chunks × vocab) matrix | bigrams, `min_df=2` |
| Retrieval | `retrieve_top_k_chunks()` | `cosine_similarity` + `argpartition` |
| Ranking | `rag_predict_row()` | scores all 5 options vs. context |
| Evaluation | MAP@3 via `apk()` / `mapk()` | per-label breakdown included |
| Submission | `submission.csv` → `/kaggle/working/` | Competition-ready format |
| Dense (bonus) | `sentence-transformers` SBERT | Needs Kaggle GPU T4 |

### Next Steps to Improve Score
1. **Enable GPU + Dense Retrieval** (Section 10) — `all-MiniLM-L6-v2` captures semantic similarity
2. **Bigger top-K** — try `top_k_retrieval=10` or `20` for hard topics
3. **Query expansion** — strip template prefixes ("Pick the best possible answer:") before retrieval
4. **Hybrid scoring** — combine TF-IDF score + dense score for best of both
5. **Cross-encoder reranking** — use a cross-encoder on top-20 to rerank to top-5